[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke06-klinisk-praksis/01_risikomodell_logistisk_regresjon_kalibrering_shap.ipynb)


# 🏥 Syntetisk risikomodell: logistisk regresjon, kalibrering og SHAP

## Læringsmål
- Trene en enkel, interpretable risikomodell
- Vurdere diskriminering (AUROC) og kalibrering (Brier/kurver)
- Forklare prediksjoner med SHAP


### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab


In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    # SHAP kan mangle i base-Colab
    try:
        import shap  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])  # type: ignore
    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])  # type: ignore
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokalt miljø")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
import shap

rng = np.random.default_rng(42)
print("✅ Miljø klart")


In [ ]:
# Syntetiske data
X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    weights=[0.7, 0.3],
    class_sep=1.0,
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)


In [ ]:
# Modell og kalibrering
base = LogisticRegression(max_iter=1000, solver='lbfgs')
base.fit(X_train, y_train)
cal = CalibratedClassifierCV(base_estimator=base, method="isotonic", cv=5)
cal.fit(X_train, y_train)

proba_test = cal.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba_test)
brier = brier_score_loss(y_test, proba_test)
print(f"AUROC: {auc:.3f}  |  Brier: {brier:.3f}")

# Kalibreringskurve
prob_true, prob_pred = calibration_curve(y_test, proba_test, n_bins=10, strategy='uniform')
import matplotlib.pyplot as plt
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1], 'k--', label='Perfekt')
plt.plot(prob_pred, prob_true, marker='o', label='Modell')
plt.xlabel('Predikert risiko')
plt.ylabel('Observ. sannsynlighet')
plt.title('Kalibreringskurve')
plt.legend(); plt.tight_layout(); plt.show()

# ROC-kurve
fpr, tpr, thr = roc_curve(y_test, proba_test)
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC'); plt.tight_layout(); plt.show()


In [ ]:
# SHAP-forklaringer
explainer = shap.Explainer(cal, X_train)
shap_values = explainer(X_test[:200])

# Global oversikt (bar)
shap.plots.bar(shap_values, max_display=10)

# Lokal forklaring (force plot) – vis første
shap.plots.force(shap_values[0])


### Refleksjon
- Når er kalibrering viktigere enn høy AUROC?
- Hvordan påvirker korrelerte variabler SHAP-tolkning?
- Hvilken terskel ville du valgt, og hvorfor?
